# exp138_ancc_surface_predictability_audit train

Fold-safe audit of train-only `ANCC` surface predictability without LightGBM.

## Contents

1. Setup and configuration
2. Input preview
3. Fold-safe ANCC surface audit
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import os

import pandas as pd

from ancc_surface_predictability_audit import load_wells, run_audit
from settings import EXPERIMENT_NAME, ExperimentPaths, load_config

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", config.get("experiment", {}).get("route"))
print("LightGBM used:", config.get("model", {}).get("uses_lightgbm"))
print("Train data:", paths.train_data_dir)
print("Artifacts:", paths.artifacts_dir)
print("Features:", paths.features_dir)
print("Debug:", DEBUG, "Max wells:", MAX_WELLS)

## 2. Input preview

In [ ]:
preview_wells = load_wells(paths.train_data_dir, max_wells=5)
preview = []
for well in preview_wells:
    frame = well.frame
    preview.append(
        {
            "well": well.well,
            "rows": len(frame),
            "eval_rows": int(well.eval_mask.sum()),
            "anchor_position": well.anchor_position,
            "ancc_missing": int(frame["ANCC"].isna().sum()),
        }
    )
pd.DataFrame(preview)

## 3. Fold-safe ANCC surface audit

In [ ]:
metrics = run_audit(
    train_dir=paths.train_data_dir,
    artifacts_dir=paths.artifacts_dir,
    features_dir=paths.features_dir,
    metrics_path=paths.metrics_path,
    config=config,
    debug=DEBUG,
    max_wells=MAX_WELLS,
)
print(json.dumps(metrics["best_by_delta_rmse"], indent=2))
print("Metrics written:", paths.metrics_path)

## 4. Metrics and artifacts

In [ ]:
method_metrics = pd.read_csv(paths.artifacts_dir / "method_metrics.csv")
bucket_metrics = pd.read_csv(paths.artifacts_dir / "bucket_metrics.csv")
target_summary = pd.read_csv(paths.artifacts_dir / "target_distribution_summary.csv")

display(method_metrics.sort_values(["delta_rmse", "rmse"]))
display(bucket_metrics.head(30))
display(target_summary[target_summary["bucket"].eq("all")].head(20))
print("OOF predictions:", paths.features_dir / "ancc_surface_oof_predictions.csv")
print("SHA256:", json.dumps(metrics["sha256"], indent=2))